In [1]:
# =========================================================
# PS S6E4 - exp_20260415_046a_xgb024_optuna_only
# Optuna tuning for 024 XGB digit orderedTE model
# feature pipeline is inherited from 024
# =========================================================

In [2]:
import os
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd
import optuna
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

In [3]:
class CFG:
    EXP_ID = "exp_20260415_046_xgb024_optuna"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    ID_COL = "id"
    TARGET = "Irrigation_Need"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    SEED = 2026
    N_SPLITS = 5
    N_TRIALS = 3

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

In [4]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)
y = train[CFG.TARGET].values

drop_cols = [CFG.ID_COL]

train = train.drop(drop_cols, axis=1)
test = test.drop(drop_cols, axis=1)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
display(train.head())

CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


In [5]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []

            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]

                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test):
        test = test.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test = test.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]

        return test

In [6]:
def build_features_024(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    024 の feature pipeline のうち、
    - digit features
    - constant-column drop
    - frequency-based category remap
    までを作る。

    OrderedTE は fold 内でかけるので、ここではまだ実行しない。

    Returns
    -------
    X_train_base : pd.DataFrame
        target を除いた base features（TE 前）
    X_test_base : pd.DataFrame
        test 側の base features（TE 前）
    meta : dict
        fold 内 TE で使うメタ情報
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    # 元 024 と同じ前提
    target_col = CFG.TARGET
    id_col = CFG.ID_COL

    # id は落とす
    if id_col in train_df.columns:
        train_df = train_df.drop(columns=[id_col])
    if id_col in test_df.columns:
        test_df = test_df.drop(columns=[id_col])

    # target は train にだけある
    CATS = [c for c in test_df.columns if train_df[c].dtype == object]
    NUMS = [c for c in test_df.columns if c not in CATS]

    # 元 024 と同じ digit FE
    M = train_df[NUMS].max()

    def add_digit_features(df):
        df = df.copy()
        for c in NUMS:
            for k in range(-4, 4):
                df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

            if M[c] < 10:
                df[c] = df[c].round(3)
            elif M[c] < 100:
                df[c] = df[c].round(2)
            else:
                df[c] = df[c].round(1)
        return df

    train_df = add_digit_features(train_df)
    test_df = add_digit_features(test_df)

    # 定数列 drop は train/test 両方を見て安全に決める
    base_test_cols = [c for c in test_df.columns if c != target_col]
    DROP = [c for c in base_test_cols if test_df[c].nunique() == 1]
    if DROP:
        train_df = train_df.drop(columns=DROP, errors="ignore")
        test_df = test_df.drop(columns=DROP, errors="ignore")

    # digit 列をカテゴリ扱い
    CATEGORY = CATS + [c for c in test_df.columns if "digit" in c]
    FEATURES = CATEGORY + NUMS

    # frequency-based remap
    for c in CATEGORY:
        freq = train_df[c].value_counts()
        mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
        mapping_default = len(mapping)

        train_df[c] = train_df[c].map(lambda x: mapping.get(x, mapping_default))
        test_df[c] = test_df[c].map(lambda x: mapping.get(x, mapping_default))

    # base features only
    X_train_base = train_df[FEATURES].copy()
    X_test_base = test_df[FEATURES].copy()

    meta = {
        "CATS": CATS,
        "NUMS": NUMS,
        "CATEGORY": CATEGORY,
        "FEATURES": FEATURES,
        "DROP": DROP,
        "target_col": target_col,
    }

    return X_train_base, X_test_base, meta


X_train_base, X_test_base, feature_meta = build_features_024(train.copy(), test.copy())


def make_xgb_params(trial: optuna.trial.Trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000, step=250),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.12, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 6.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.70, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 2.0),
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": CFG.SEED,
        "n_jobs": -1,
        "verbosity": 0,
    }


def cv_score_xgb(params, X_base, y, X_test_base, feature_meta, trial=None):
    kf = KFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=42)
    oof = np.zeros((len(X_base), 3), dtype=np.float32)
    fold_scores = []

    CATEGORY = feature_meta["CATEGORY"]
    CATS = feature_meta["CATS"]
    target_col = CFG.TARGET

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_base, y), 1):
        X_tr_base = X_base.iloc[tr_idx].copy()
        X_va_base = X_base.iloc[va_idx].copy()

        y_tr = pd.Series(y[tr_idx], name=target_col)
        y_va = y[va_idx]

        unique, counts = np.unique(y_tr.values, return_counts=True)
        count_dict = dict(zip(unique, counts))
        avg_count = len(y_tr) / len(unique)
        weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
        train_weights = np.array([weights_dict[v] for v in y_tr.values], dtype=np.float32)

        te = OrderedTE(a=1)

        full_df = pd.concat([X_tr_base.reset_index(drop=True), y_tr.reset_index(drop=True)], axis=1)
        full_df["weight"] = train_weights

        te_train = te.fit(
            full_df.sample(frac=1, random_state=42),
            category_cols=CATEGORY,
            target_col=target_col,
        )

        X_tr = te_train.drop([target_col, "weight"], axis=1)
        y_tr_fit = te_train[target_col].values
        train_weights_fit = te_train["weight"].values

        X_va = te.transform(X_va_base.reset_index(drop=True))

        X_tr = X_tr.drop(columns=CATS, errors="ignore")
        X_va = X_va.drop(columns=CATS, errors="ignore")

        model = xgb.XGBClassifier(
            **params,
            early_stopping_rounds=100,
        )

        model.fit(
            X_tr,
            y_tr_fit,
            sample_weight=train_weights_fit,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        va_pred = model.predict_proba(X_va)
        oof[va_idx] = va_pred

        sc = float(balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1)))
        fold_scores.append(sc)
        print(f"fold {fold} BA: {sc:.6f}")

        if trial is not None:
            trial.report(np.mean(fold_scores), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        del model, X_tr, X_va, y_tr_fit, train_weights_fit, va_pred
        gc.collect()

    full_score = float(balanced_accuracy_score(y, np.argmax(oof, axis=1)))
    return full_score, fold_scores, oof


def objective(trial: optuna.trial.Trial):
    params = make_xgb_params(trial)
    score, fold_scores, _ = cv_score_xgb(params, X_train_base, y, X_test_base, feature_meta, trial=trial)
    trial.set_user_attr("fold_scores", fold_scores)
    return score


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=1, n_warmup_steps=1),
)
study.optimize(objective, n_trials=CFG.N_TRIALS, show_progress_bar=True)

[I 2026-04-15 23:50:22,934] A new study created in memory with name: no-name-07140c0d-4b2d-4a6d-a90f-f51586f0c361


  0%|          | 0/3 [00:00<?, ?it/s]

fold 1 BA: 0.970355
fold 2 BA: 0.972019
fold 3 BA: 0.970368
fold 4 BA: 0.973246
fold 5 BA: 0.970456
[I 2026-04-16 00:40:45,544] Trial 0 finished with value: 0.9712857881577435 and parameters: {'n_estimators': 750, 'max_depth': 5, 'learning_rate': 0.03977533040009379, 'min_child_weight': 2.325249048935525, 'subsample': 0.7323943671724104, 'colsample_bytree': 0.7029950347053535, 'gamma': 1.4147462693479202, 'reg_alpha': 5.883405699162615e-05, 'reg_lambda': 0.006694001802770697, 'max_delta_step': 1.9241578616301294}. Best is trial 0 with value: 0.9712857881577435.
fold 1 BA: 0.968477
[I 2026-04-16 00:53:00,320] Trial 1 pruned. 
fold 1 BA: 0.966910
[I 2026-04-16 00:58:17,867] Trial 2 pruned. 


In [7]:
best_params = study.best_params
best_value = study.best_value
best_trial_number = study.best_trial.number
best_fold_scores = study.best_trial.user_attrs.get("fold_scores", None)

print("best_value:", best_value)
print("best_params:", best_params)
print("best_trial_number:", best_trial_number)

trials_df = study.trials_dataframe()
trials_df.to_csv(f"{CFG.OUTPUT_DIR}/optuna_trials.csv", index=False)

best_params_payload = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260409_024_xgb_digit_orderedte_min",
    "best_trial_number": int(best_trial_number),
    "best_cv_raw": float(best_value),
    "best_fold_scores": best_fold_scores,
    "best_params": best_params,
    "n_trials": CFG.N_TRIALS,
    "notes": {
        "phase": "optuna_only",
        "bias_not_applied_yet": True,
        "feature_pipeline": "same_as_024",
        "ordered_te_in_fold": True,
    },
}
save_json(best_params_payload, f"{CFG.OUTPUT_DIR}/best_params.json")
save_json(best_params_payload, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("saved optuna-only outputs to:", CFG.OUTPUT_DIR)

best_value: 0.9712857881577435
best_params: {'n_estimators': 750, 'max_depth': 5, 'learning_rate': 0.03977533040009379, 'min_child_weight': 2.325249048935525, 'subsample': 0.7323943671724104, 'colsample_bytree': 0.7029950347053535, 'gamma': 1.4147462693479202, 'reg_alpha': 5.883405699162615e-05, 'reg_lambda': 0.006694001802770697, 'max_delta_step': 1.9241578616301294}
best_trial_number: 0
saved optuna-only outputs to: /kaggle/working/exp_20260415_046_xgb024_optuna
